In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

Cloning into 'CTAB-GAN-Plus'...


In [3]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

from model.ctabgan import CTABGAN

# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
breast_cancer = fetch_ucirepo(id=17)

X = breast_cancer.data.features
y = breast_cancer.data.targets

cancer_data = pd.concat([X, y], axis=1)

target_col = cancer_data.columns[-1]

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(cancer_data)

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []

In [4]:
# ---------------------------------------------------
# SINGLE RUN
# ---------------------------------------------------

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ---------------------------------------------------
# TRAIN / TEST SPLIT (NO LEAKAGE)
# ---------------------------------------------------

train_real, test_real = train_test_split(
    cancer_data,
    test_size=TEST_SIZE,
    stratify=cancer_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# CTABGAN
# ---------------------------------------------------

try:

    data_path = "breast_cancer_train.csv"
    train_real.to_csv(data_path, index=False)

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=[target_col],
        log_columns=[],
        mixed_columns={},
        integer_columns=[],
        problem_type={"Classification": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)


================ SINGLE RUN ================


100%|██████████| 150/150 [10:36<00:00,  4.25s/it]


Finished training in 644.74285364151  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 458.97it/s]|
Column Shapes Score: 84.89%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:01<00:00, 277.12it/s]|
Column Pair Trends Score: 88.39%

Overall Score (Average): 86.64%

CTABGAN: 0.8664


In [5]:
# WGAN-GP

try:

    import traceback

    data_wgan = train_real.copy()

    encoder = LabelEncoder()
    data_wgan[target_col] = encoder.fit_transform(data_wgan[target_col])

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    synthetic_wgan[target_col] = (
        synthetic_wgan[target_col]
        .round()
        .clip(0, 1)
        .astype(int)
    )

    synthetic_wgan[target_col] = encoder.inverse_transform(
        synthetic_wgan[target_col]
    )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 591.62it/s]|
Column Shapes Score: 87.61%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:01<00:00, 310.53it/s]|
Column Pair Trends Score: 91.85%

Overall Score (Average): 89.73%

WGAN_GP: 0.8973


In [6]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 480.54it/s]|
Column Shapes Score: 75.11%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:01<00:00, 300.69it/s]|
Column Pair Trends Score: 79.23%

Overall Score (Average): 77.17%

CTGAN: 0.7717
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 379.63it/s]|
Column Shapes Score: 68.38%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:01<00:00, 234.34it/s]|
Column Pair Trends Score: 78.93%

Overall Score (Average): 73.65%

CopulaGAN: 0.7365
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 638.52it/s]|
Column Shapes Score: 84.26%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:01<00:00, 357.97it/s]|
Column Pair Trends Score: 90.01%

Overall Score (Average): 87.13%

TVAE: 0.8713
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 945.82it/s]|
Column S

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import pandas as pd

In [12]:
# TRTR (Train Real, Test Real)

print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")

trtr_results = []

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"Running {model_name}...")

    for seed in SEEDS:

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed
        )

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(
            accuracy_score(y_test_real, y_pred)
        )

        f1_scores.append(
            f1_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        precision_scores.append(
            precision_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

    acc_mean = np.mean(accuracy_scores)
    acc_std = np.std(accuracy_scores)

    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores)

    prec_mean = np.mean(precision_scores)
    prec_std = np.std(precision_scores)

    rec_mean = np.mean(recall_scores)
    rec_std = np.std(recall_scores)

    trtr_results.append({
        "Model": model_name,
        "Accuracy (Mean±Std)_TRTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
        "F1 (Mean±Std)_TRTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
        "Precision (Mean±Std)_TRTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
        "Recall (Mean±Std)_TRTR": f"{rec_mean:.4f} ± {rec_std:.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results)

display(
    trtr_results_df[
        [
            "Model",
            "Accuracy (Mean±Std)_TRTR",
            "F1 (Mean±Std)_TRTR",
            "Precision (Mean±Std)_TRTR",
            "Recall (Mean±Std)_TRTR"
        ]
    ]
)

--- Starting TRTR Evaluation (Train Real, Test Real) ---
Running LogReg...
Running SVM-RBF...
Running KNN...
Running NaiveBayes...
Running DecisionTree...
Running RandomForest...
Running ExtraTrees...
Running GradientBoost...
Running AdaBoost...
Running MLP...


,Model,Accuracy (Mean±Std)_TRTR,F1 (Mean±Std)_TRTR,Precision (Mean±Std)_TRTR,Recall (Mean±Std)_TRTR
0,LogReg,0.9500 ± 0.0204,0.9498 ± 0.0206,0.9506 ± 0.0205,0.9500 ± 0.0204
1,SVM-RBF,0.9132 ± 0.0209,0.9112 ± 0.0215,0.9177 ± 0.0216,0.9132 ± 0.0209
2,KNN,0.9228 ± 0.0235,0.9223 ± 0.0237,0.9236 ± 0.0237,0.9228 ± 0.0235
3,NaiveBayes,0.9395 ± 0.0216,0.9388 ± 0.0219,0.9410 ± 0.0217,0.9395 ± 0.0216
4,DecisionTree,0.9351 ± 0.0205,0.9351 ± 0.0205,0.9362 ± 0.0197,0.9351 ± 0.0205
5,RandomForest,0.9623 ± 0.0136,0.9622 ± 0.0136,0.9627 ± 0.0137,0.9623 ± 0.0136
6,ExtraTrees,0.9702 ± 0.0119,0.9701 ± 0.0120,0.9707 ± 0.0121,0.9702 ± 0.0119
7,GradientBoost,0.9632 ± 0.0203,0.9631 ± 0.0203,0.9637 ± 0.0201,0.9632 ± 0.0203
8,AdaBoost,0.9667 ± 0.0102,0.9665 ± 0.0103,0.9673 ± 0.0100,0.9667 ± 0.0102
9,MLP,0.9167 ± 0.0286,0.9161 ± 0.0281,0.9205 ± 0.0261,0.9167 ± 0.0286


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=[42,43,44,45,46,47,48,49,50,51]
):

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_train, _, y_train, _ = train_test_split(
                X_train,
                y_train,
                test_size=test_size,
                random_state=seed,
                stratify=y_train
            )

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            _, X_test, _, y_test = train_test_split(
                X_test,
                y_test,
                test_size=test_size,
                random_state=seed,
                stratify=y_test
            )

            scaler = StandardScaler().fit(X_train)

            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test,
                    y_pred,
                    pos_label="M",
                    average="binary",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test,
                    y_pred,
                    pos_label="M",
                    average="binary",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test,
                    y_pred,
                    pos_label="M",
                    average="binary",
                    zero_division=0
                )
            )

        results.append({
            "Model": name,

            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),

            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),

            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),

            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),

            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )

In [19]:
import pandas as pd

label_col = "Diagnosis"

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "WGAN_GP",
    "CTABGAN"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=cancer_data,
    test_df=cancer_data,
    label="diagnosis",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=cancer_data,
        label="diagnosis",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
0,LogReg,0.9789 ± 0.0098,0.9708 ± 0.0138,0.9902 ± 0.0120,0.9524 ± 0.0213
9,MLP,0.9746 ± 0.0133,0.9646 ± 0.0187,0.9855 ± 0.0192,0.9452 ± 0.0302
1,SVM-RBF,0.9702 ± 0.0189,0.9584 ± 0.0269,0.9755 ± 0.0193,0.9429 ± 0.0442
6,ExtraTrees,0.9702 ± 0.0119,0.9590 ± 0.0165,0.9715 ± 0.0251,0.9476 ± 0.0278
8,AdaBoost,0.9667 ± 0.0102,0.9538 ± 0.0144,0.9735 ± 0.0221,0.9357 ± 0.0283
7,GradientBoost,0.9632 ± 0.0203,0.9494 ± 0.0278,0.9620 ± 0.0358,0.9381 ± 0.0340
2,KNN,0.9623 ± 0.0188,0.9471 ± 0.0267,0.9751 ± 0.0246,0.9214 ± 0.0399
5,RandomForest,0.9623 ± 0.0136,0.9482 ± 0.0184,0.9618 ± 0.0297,0.9357 ± 0.0214
4,DecisionTree,0.9351 ± 0.0205,0.9122 ± 0.0277,0.9097 ± 0.0373,0.9167 ± 0.0416
3,NaiveBayes,0.9325 ± 0.0232,0.9056 ± 0.0328,0.9310 ± 0.0364,0.8833 ± 0.0516


CTGAN - TSTR


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
3,NaiveBayes,0.8518 ± 0.0295,0.8125 ± 0.0341,0.7650 ± 0.0485,0.8690 ± 0.0416
0,LogReg,0.7912 ± 0.0129,0.7658 ± 0.0116,0.6536 ± 0.0192,0.9262 ± 0.0310
1,SVM-RBF,0.7342 ± 0.0239,0.7058 ± 0.0202,0.5974 ± 0.0271,0.8643 ± 0.0302
8,AdaBoost,0.7254 ± 0.0251,0.7066 ± 0.0262,0.5843 ± 0.0259,0.9000 ± 0.0744
9,MLP,0.7202 ± 0.0669,0.7179 ± 0.0486,0.5814 ± 0.0688,0.9500 ± 0.0457
5,RandomForest,0.7158 ± 0.0272,0.6894 ± 0.0307,0.5773 ± 0.0260,0.8571 ± 0.0553
6,ExtraTrees,0.7035 ± 0.0271,0.6863 ± 0.0268,0.5632 ± 0.0255,0.8810 ± 0.0532
7,GradientBoost,0.6956 ± 0.0510,0.6804 ± 0.0354,0.5599 ± 0.0455,0.8738 ± 0.0564
4,DecisionTree,0.6421 ± 0.0467,0.5725 ± 0.0748,0.5141 ± 0.0541,0.6667 ± 0.1456
2,KNN,0.6140 ± 0.0541,0.6243 ± 0.0360,0.4900 ± 0.0402,0.8667 ± 0.0604


CTGAN - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
0,LogReg,0.9123 ± 0.0188,0.8737 ± 0.0285,0.9284 ± 0.0416,0.8286 ± 0.0561
6,ExtraTrees,0.9061 ± 0.0118,0.8671 ± 0.0183,0.9071 ± 0.0365,0.8333 ± 0.0452
5,RandomForest,0.9053 ± 0.0228,0.8640 ± 0.0315,0.9207 ± 0.0531,0.8167 ± 0.0427
8,AdaBoost,0.9018 ± 0.0195,0.8559 ± 0.0318,0.9261 ± 0.0221,0.7976 ± 0.0556
7,GradientBoost,0.8991 ± 0.0261,0.8522 ± 0.0386,0.9271 ± 0.0531,0.7905 ± 0.0474
1,SVM-RBF,0.8965 ± 0.0225,0.8502 ± 0.0335,0.9095 ± 0.0369,0.8000 ± 0.0490
2,KNN,0.8939 ± 0.0194,0.8454 ± 0.0277,0.9157 ± 0.0528,0.7881 ± 0.0419
3,NaiveBayes,0.8904 ± 0.0193,0.8385 ± 0.0306,0.9145 ± 0.0339,0.7762 ± 0.0490
9,MLP,0.8640 ± 0.0249,0.8041 ± 0.0296,0.8698 ± 0.0791,0.7548 ± 0.0511
4,DecisionTree,0.7579 ± 0.0434,0.6694 ± 0.0540,0.6795 ± 0.0737,0.6643 ± 0.0643


CTABGAN - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,CTABGAN,LogReg,0.066667,0.097046,0.061727,0.123810,0.9789 ± 0.0098,0.9123 ± 0.0188
1,CTABGAN,MLP,0.110526,0.160488,0.115714,0.190476,0.9746 ± 0.0133,0.8640 ± 0.0249
2,CTABGAN,SVM-RBF,0.073684,0.108225,0.066018,0.142857,0.9702 ± 0.0189,0.8965 ± 0.0225
3,CTABGAN,ExtraTrees,0.064035,0.091911,0.064445,0.114286,0.9702 ± 0.0119,0.9061 ± 0.0118
4,CTABGAN,AdaBoost,0.064912,0.097908,0.047490,0.138095,0.9667 ± 0.0102,0.9018 ± 0.0195
5,CTABGAN,GradientBoost,0.064035,0.097190,0.034910,0.147619,0.9632 ± 0.0203,0.8991 ± 0.0261
6,CTABGAN,KNN,0.068421,0.101664,0.059475,0.133333,0.9623 ± 0.0188,0.8939 ± 0.0194
7,CTABGAN,RandomForest,0.057018,0.084185,0.041100,0.119048,0.9623 ± 0.0136,0.9053 ± 0.0228
8,CTABGAN,DecisionTree,0.177193,0.242895,0.230247,0.252381,0.9351 ± 0.0205,0.7579 ± 0.0434
9,CTABGAN,NaiveBayes,0.042105,0.067056,0.016569,0.107143,0.9325 ± 0.0232,0.8904 ± 0.0193


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
5,WGAN_GP,0.022193,0.034613,-0.006913,0.069524
4,TVAE,0.035175,0.043345,0.082329,0.000952
0,CTABGAN,0.078860,0.114857,0.073770,0.146905
3,GaussianCopula,0.101140,0.153587,0.067085,0.213571
1,CTGAN,0.242193,0.250756,0.374961,0.066429
2,CopulaGAN,0.602281,0.532819,0.647487,0.308810


In [20]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
